In [16]:
from harbor.analysis import cross_docking as cd
from importlib import reload
reload(cd)

<module 'harbor.analysis.cross_docking' from '/Users/alexpayne/Scientific_Projects/harbor/harbor/analysis/cross_docking.py'>

In [17]:
from pathlib import Path
import pandas as pd

# Imports

In [362]:
source_path = Path("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/20250212_p_to_x_posit/")

In [363]:
df = pd.read_csv(source_path / "20250311_combined_results")

## try saving to parquet instead

In [364]:
df.to_parquet(source_path / "20250311_combined_results.parquet")

In [366]:
df = pd.read_parquet(source_path / "20250311_combined_results.parquet")

In [348]:
# len(df)

In [349]:
# df = df[(df.bitsize == 2048)&(df.radius == 2.0)]

In [350]:
# len(df)

In [351]:
# df.to_csv(source_path / "20250311_ecfp4_only.csv")

In [352]:
# df = pd.read_csv(source_path / "20250311_ecfp4_only.csv", index_col=0)

In [389]:
reload(cd)
sso = cd.ScaffoldSplitOptions
ev = cd.Evaluator(dataset_split=cd.ScaffoldSplit(query_scaffold_id_column='cluster_id',
                                             reference_scaffold_id_column='cluster_id_Reference',      
                                             split_option=sso.X_TO_NOT_X,
                                                 query_scaffold_id_subset=[0],
                                             n_per_split=-1),
                  scorer=cd.POSITScorer,
                  evaluator=cd.BinaryEvaluation(variable="RMSD", cutoff=2),
                  groupby=['Query_Ligand'],
                  n_bootstraps=1000)

In [390]:
results = ev.run(df)

In [391]:
results

FractionGood(type_='FractionGood', name='FractionGood', total=66, fraction=0.9545454545454546, replicates=[0.9545454545454546])

# explore similarity split

In [354]:
reload(cd)
sim_split = cd.SimilaritySplit(n_per_split=100,
                                                   similarity_column='Tanimoto',
                                                   query_ligand_column='Query_Ligand',
                                                   groupby={'Aligned': True, 'Type': 'TanimotoCombo'},
                                                   include_similar=False
                                                   )
ev = cd.Evaluator(dataset_split=sim_split,
                                                   scorer=cd.POSITScorer,
                                                   evaluator=cd.BinaryEvaluation(variable="RMSD", cutoff=2),
                  n_bootstraps=100,
                  groupby=['Query_Ligand'])

In [355]:
cd.SimilaritySplit(n_per_split=-1,
                                                   similarity_column='Tanimoto',
                                                   query_ligand_column='Query_Ligand',
                                                   groupby={'Aligned': True, 'Type': 'TanimotoCombo'},
                                                   include_similar=False
                                                   ).run(df)

[         Unnamed: 0        Query_Ligand Reference_Structure  \
 110             110  ERI-UCB-d6de1f3c-2      Mpro-x11368_0A   
 111             111  ERI-UCB-d6de1f3c-2      Mpro-x11368_0A   
 132             132  ERI-UCB-d6de1f3c-2      Mpro-x12300_0A   
 133             133  ERI-UCB-d6de1f3c-2      Mpro-x12300_0A   
 154             154  ERI-UCB-d6de1f3c-2      Mpro-x12300_0A   
 ...             ...                 ...                 ...   
 5599363     5599363  MAT-POS-e6dd326d-8      Mpro-x12300_0A   
 5599374     5599374  MAT-POS-e6dd326d-8      Mpro-x12300_0A   
 5599385     5599385  MAT-POS-e6dd326d-8      Mpro-x12300_0A   
 5599396     5599396  MAT-POS-e6dd326d-8      Mpro-x12300_0A   
 5599407     5599407  MAT-POS-e6dd326d-8      Mpro-x12300_0A   
 
                             Reference_Ligand_SMILES  \
 110                    Cc1ccncc1NC(=O)Cc2cccc(c2)Br   
 111                    Cc1ccncc1NC(=O)Cc2cccc(c2)Br   
 132      Cc1ccncc1NC(=O)Cc2cc(cc(c2)Cl)NCC(C)(C)C#N   
 133  

In [356]:
n_per_split=-1
similarity_column='Tanimoto'
query_ligand_column='Query_Ligand'
groupby={'Aligned': True, 'Type': 'TanimotoCombo'}
include_similar=False
higher_is_more_similar=True
threshold=0.5

# first just get the necessary data
for key, value in groupby.items():
    df = df[df[key] == value]

In [357]:
df

,Unnamed: 0,Query_Ligand,Reference_Structure,Reference_Ligand_SMILES,SMILES,docking-confidence-POSIT,RMSD,Pose_ID,POSIT_Method,Reference_Ligand,...,bitsize,fingerprint,compound_name,cluster_id,scaffold_smarts,cluster_type,compound_name_Reference,cluster_id_Reference,scaffold_smarts_Reference,cluster_type_Reference
0,0,ERI-UCB-d6de1f3c-2,Mpro-x11810_0A,c1ccc2c(c1)cncc2NC(=O)Cc3cc(ccn3)Cl,c1ccc2c(c1)cncc2C(=O)N3CCN(C(=O)C3)c4cccc(c4)Cl,0.72,1.427564,0,SHAPEFIT,PET-UNK-3c72d439-1,...,NaN,NaN,ERI-UCB-d6de1f3c-2,48,CC1CC(C(C)C2CCCC3CCCCC32)CCC1C1CCCCC1,generic_bemis_murko,PET-UNK-3c72d439-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko
1,1,ERI-UCB-d6de1f3c-2,Mpro-x11810_0A,c1ccc2c(c1)cncc2NC(=O)Cc3cc(ccn3)Cl,c1ccc2c(c1)cncc2C(=O)N3CCN(C(=O)C3)c4cccc(c4)Cl,0.72,1.427564,0,SHAPEFIT,PET-UNK-3c72d439-1,...,NaN,NaN,ERI-UCB-d6de1f3c-2,48,CC1CC(C(C)C2CCCC3CCCCC32)CCC1C1CCCCC1,generic_bemis_murko,PET-UNK-3c72d439-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko
22,22,ERI-UCB-d6de1f3c-2,Mpro-x11810_0A,c1ccc2c(c1)cncc2NC(=O)Cc3cc(ccn3)Cl,c1ccc2c(c1)cncc2C(=O)N3CCN(C(=O)C3)c4cccc(c4)Cl,0.51,2.343900,12,SHAPEFIT,PET-UNK-3c72d439-1,...,NaN,NaN,ERI-UCB-d6de1f3c-2,48,CC1CC(C(C)C2CCCC3CCCCC32)CCC1C1CCCCC1,generic_bemis_murko,PET-UNK-3c72d439-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko
23,23,ERI-UCB-d6de1f3c-2,Mpro-x11810_0A,c1ccc2c(c1)cncc2NC(=O)Cc3cc(ccn3)Cl,c1ccc2c(c1)cncc2C(=O)N3CCN(C(=O)C3)c4cccc(c4)Cl,0.51,2.343900,12,SHAPEFIT,PET-UNK-3c72d439-1,...,NaN,NaN,ERI-UCB-d6de1f3c-2,48,CC1CC(C(C)C2CCCC3CCCCC32)CCC1C1CCCCC1,generic_bemis_murko,PET-UNK-3c72d439-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko
44,44,ERI-UCB-d6de1f3c-2,Mpro-x11810_0A,c1ccc2c(c1)cncc2NC(=O)Cc3cc(ccn3)Cl,c1ccc2c(c1)cncc2C(=O)N3CCN(C(=O)C3)c4cccc(c4)Cl,0.51,2.172558,13,SHAPEFIT,PET-UNK-3c72d439-1,...,NaN,NaN,ERI-UCB-d6de1f3c-2,48,CC1CC(C(C)C2CCCC3CCCCC32)CCC1C1CCCCC1,generic_bemis_murko,PET-UNK-3c72d439-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5599374,5599374,MAT-POS-e6dd326d-8,Mpro-x12300_0A,Cc1ccncc1NC(=O)Cc2cc(cc(c2)Cl)NCC(C)(C)C#N,C=CC(=O)NC[C@]1(CCOc2c1cc(cc2)Cl)C(=O)Nc3cncc4...,0.24,2.773062,12,FRED,MAT-POS-044491d2-7,...,NaN,NaN,MAT-POS-e6dd326d-8,0,CC(CC1CCCC2CCCCC21)C1CCCC2CCCCC21,generic_bemis_murko,MAT-POS-044491d2-7,1,CC(CC1CCCCC1)CC1CCCCC1,generic_bemis_murko
5599385,5599385,MAT-POS-e6dd326d-8,Mpro-x12300_0A,Cc1ccncc1NC(=O)Cc2cc(cc(c2)Cl)NCC(C)(C)C#N,C=CC(=O)NC[C@]1(CCOc2c1cc(cc2)Cl)C(=O)Nc3cncc4...,0.24,2.497216,19,FRED,MAT-POS-044491d2-7,...,NaN,NaN,MAT-POS-e6dd326d-8,0,CC(CC1CCCC2CCCCC21)C1CCCC2CCCCC21,generic_bemis_murko,MAT-POS-044491d2-7,1,CC(CC1CCCCC1)CC1CCCCC1,generic_bemis_murko
5599396,5599396,MAT-POS-e6dd326d-8,Mpro-x12300_0A,Cc1ccncc1NC(=O)Cc2cc(cc(c2)Cl)NCC(C)(C)C#N,C=CC(=O)NC[C@]1(CCOc2c1cc(cc2)Cl)C(=O)Nc3cncc4...,0.05,6.451884,28,FRED,MAT-POS-044491d2-7,...,NaN,NaN,MAT-POS-e6dd326d-8,0,CC(CC1CCCC2CCCCC21)C1CCCC2CCCCC21,generic_bemis_murko,MAT-POS-044491d2-7,1,CC(CC1CCCCC1)CC1CCCCC1,generic_bemis_murko
5599407,5599407,MAT-POS-e6dd326d-8,Mpro-x12300_0A,Cc1ccncc1NC(=O)Cc2cc(cc(c2)Cl)NCC(C)(C)C#N,C=CC(=O)NC[C@]1(CCOc2c1cc(cc2)Cl)C(=O)Nc3cncc4...,0.05,6.449110,33,FRED,MAT-POS-044491d2-7,...,NaN,NaN,MAT-POS-e6dd326d-8,0,CC(CC1CCCC2CCCCC21)C1CCCC2CCCCC21,generic_bemis_murko,MAT-POS-044491d2-7,1,CC(CC1CCCCC1)CC1CCCCC1,generic_bemis_murko


In [337]:
# if include similar True and higher is MORE similar, or if similar False and higher is LESS similar
if include_similar == higher_is_more_similar:
    df = df[df[similarity_column] >= threshold]

# if include similar True and higher is LESS similar, or if similar False and higher is MORE similar
elif include_similar != higher_is_more_similar:
    df = df[df[similarity_column] <= threshold]

# finally, group by the query ligand column and randomly sample from the top N structures
df = df.groupby(query_ligand_column).head(n_per_split)

,Unnamed: 0,Query_Ligand,Reference_Structure,Reference_Ligand_SMILES,SMILES,docking-confidence-POSIT,RMSD,Pose_ID,POSIT_Method,Reference_Ligand,...,bitsize,fingerprint,compound_name,cluster_id,scaffold_smarts,cluster_type,compound_name_Reference,cluster_id_Reference,scaffold_smarts_Reference,cluster_type_Reference


# scaffold split

In [396]:
reload(cd)
sso = cd.ScaffoldSplitOptions
ev = cd.Evaluator(dataset_split=cd.ScaffoldSplit(query_scaffold_id_column='cluster_id',
                                             reference_scaffold_id_column='cluster_id_Reference',      
                                             split_option=sso.NOT_X_TO_X,
                                                 # query_scaffold_id_subset=[0],
                                                 # reference_scaffold_id_subset=[0],
                                             n_per_split=-1),
                  scorer=cd.POSITScorer,
                  evaluator=cd.BinaryEvaluation(variable="RMSD", cutoff=2),
                  groupby=['Query_Ligand'],
                  n_bootstraps=1000)
results = ev.run(df)

ValidationError: 1 validation error for ScaffoldSplit
  Value error, ScaffoldSplitOptions.NOT_X_TO_X requires exactly one reference scaffold item [type=value_error, input_value={'query_scaffold_id_colum...: 4>, 'n_per_split': -1}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.6/v/value_error

In [393]:
evs = []
for ref_cluster_id in df.cluster_id_Reference.unique():
    if df[df.cluster_id_Reference == ref_cluster_id].groupby('Reference_Ligand').count().shape[0] > 5:
        evs.append(cd.Evaluator(dataset_split=cd.ScaffoldSplit(query_scaffold_id_column='cluster_id',
                                             reference_scaffold_id_column='cluster_id_Reference',      
                                             split_option=sso.NOT_X_TO_X,
                                                 # query_scaffold_id_subset=[0],
                                                 reference_scaffold_id_subset=[ref_cluster_id],
                                             n_per_split=5),
                  scorer=cd.POSITScorer,
                  evaluator=cd.BinaryEvaluation(variable="RMSD", cutoff=2),
                  groupby=['Query_Ligand'],
                  n_bootstraps=1000))

In [414]:
subset = [ref_cluster_id for ref_cluster_id in df.cluster_id_Reference.unique() if df[df.cluster_id_Reference == ref_cluster_id].groupby('Reference_Ligand').count().shape[0] > 5]

In [415]:
subset

[2, 1, 11, 3, 0, 6, 4, 7, 10, 8, 12]

In [418]:
# Get cluster sizes by counting unique ligands per cluster
cluster_sizes = df.groupby('cluster_id_Reference')['Reference_Ligand'].nunique()

# Filter for clusters with more than 5 members
subset_v2 = cluster_sizes[cluster_sizes > 5].index.tolist()

In [419]:
subset_v2

[0, 1, 2, 3, 4, 6, 7, 8, 10, 11, 12]

In [394]:
df.groupby('Query_Ligand').sample(10, replace=True)

,Unnamed: 0,Query_Ligand,Reference_Structure,Reference_Ligand_SMILES,SMILES,docking-confidence-POSIT,RMSD,Pose_ID,POSIT_Method,Reference_Ligand,...,bitsize,fingerprint,compound_name,cluster_id,scaffold_smarts,cluster_type,compound_name_Reference,cluster_id_Reference,scaffold_smarts_Reference,cluster_type_Reference
374504,374504,ADA-UCB-6c2cb422-1,Mpro-x11318_0A,Cc1c(cncc1NC(=O)Cc2cccc(c2)C#N)N,c1ccc2c(c1)cncc2NC(=O)Cc3cccc(c3)Cl,0.69,7.387604,15,SHAPEFIT,TRY-UNI-714a760b-19,...,NaN,None,ADA-UCB-6c2cb422-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko,TRY-UNI-714a760b-19,1,CC(CC1CCCCC1)CC1CCCCC1,generic_bemis_murko
350416,350416,ADA-UCB-6c2cb422-1,Mpro-x1425_0A,CC(=O)N1CCN(CC1)c2ccc(cc2)OC,c1ccc2c(c1)cncc2NC(=O)Cc3cccc(c3)Cl,0.05,6.452305,11,FRED,AAR-POS-0daf6b7e-18,...,NaN,None,ADA-UCB-6c2cb422-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko,AAR-POS-0daf6b7e-18,21,C1CCC(C2CCCCC2)CC1,generic_bemis_murko
383667,383667,ADA-UCB-6c2cb422-1,Mpro-x0874_0A,c1cscc1[C@@H]2CCC[C@@H]2C(=O)N,c1ccc2c(c1)cncc2NC(=O)Cc3cccc(c3)Cl,0.05,6.237171,34,FRED,AAR-POS-d2a4d1df-14,...,1024.0,ECFP8_1024,ADA-UCB-6c2cb422-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko,AAR-POS-d2a4d1df-14,93,C1CCC(C2CCCC2)C1,generic_bemis_murko
372859,372859,ADA-UCB-6c2cb422-1,Mpro-x10871_0A,CCC(=O)Nc1ccc(cc1)N(Cc2ccsc2)C(=O)Cn3c4ccccc4nn3,c1ccc2c(c1)cncc2NC(=O)Cc3cccc(c3)Cl,0.24,7.782280,10,FRED,ALP-POS-c59291d4-2,...,2048.0,ECFP4_2048,ADA-UCB-6c2cb422-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko,ALP-POS-c59291d4-2,14,CC(CC1CCC2CCCCC21)C(CC1CCCC1)C1CCCCC1,generic_bemis_murko
390913,390913,ADA-UCB-6c2cb422-1,Mpro-x10626_0A,Cc1ccncc1NC(=O)C2[C@H]3[C@@H]2CCCC3,c1ccc2c(c1)cncc2NC(=O)Cc3cccc(c3)Cl,0.33,7.718190,24,HYBRID,MAT-POS-590ac91e-19,...,2048.0,ECFP4_2048,ADA-UCB-6c2cb422-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko,MAT-POS-590ac91e-19,116,CC(CC1CCCCC1)C1C2CCCCC21,generic_bemis_murko
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2832999,2832999,VLA-UNK-82501c2c-1,Mpro-x1425_0A,CC(=O)N1CCN(CC1)c2ccc(cc2)OC,c1cc(c(cc1CC(=O)Nc2cncc3c2ccnc3)Cl)Cl,0.02,7.809197,10,FRED,AAR-POS-0daf6b7e-18,...,1024.0,ECFP4_1024,VLA-UNK-82501c2c-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko,AAR-POS-0daf6b7e-18,21,C1CCC(C2CCCCC2)CC1,generic_bemis_murko
2831360,2831360,VLA-UNK-82501c2c-1,Mpro-x3110_0B,CCC(=O)N(c1ccc(cc1)C(C)(C)C)[C@H](c2cccnc2)C(=...,c1cc(c(cc1CC(=O)Nc2cncc3c2ccnc3)Cl)Cl,0.05,6.761538,16,FRED,LON-WEI-adc59df6-2,...,2048.0,ECFP8_2048,VLA-UNK-82501c2c-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko,LON-WEI-adc59df6-2,12,C1CCC(CCC2CCCCC2)CC1,generic_bemis_murko
2842063,2842063,VLA-UNK-82501c2c-1,Mpro-x1374_0A,Cc1ccc(cc1)N([C@@H]2CCS(=O)(=O)C2)C(=O)C,c1cc(c(cc1CC(=O)Nc2cncc3c2ccnc3)Cl)Cl,0.02,3.922788,38,FRED,AAR-POS-0daf6b7e-7,...,1024.0,ECFP4_1024,VLA-UNK-82501c2c-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko,AAR-POS-0daf6b7e-7,30,CC1(C)CCC(CC2CCCCC2)C1,generic_bemis_murko
2842772,2842772,VLA-UNK-82501c2c-1,Mpro-x10876_0A,CN(C)c1ccc(cc1)N(Cc2ccsc2)C(=O)Cn3c4ccccc4nn3,c1cc(c(cc1CC(=O)Nc2cncc3c2ccnc3)Cl)Cl,0.24,4.601254,17,FRED,ALP-POS-d2866bdf-1,...,2048.0,ECFP8_2048,VLA-UNK-82501c2c-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko,ALP-POS-d2866bdf-1,14,CC(CC1CCC2CCCCC21)C(CC1CCCC1)C1CCCCC1,generic_bemis_murko


In [261]:
many_results = [cd.Results(evaluator=ev, fraction_good=ev.run(df)) for ev in evs]

In [263]:
results_df = cd.Results.df_from_results(many_results)

In [295]:
results_df['Scaffold'] = results_df['Reference_Scaffold_ID_Subset'].apply(lambda x: x[0])

In [296]:
def get_scaffold_counts(df, q, scaff):
    return df.groupby([q, scaff]).head(1).groupby(scaff).count()[[q]]

ref_scaff_count = get_scaffold_counts(df, 'Reference_Ligand', 'cluster_id_Reference')

In [297]:
ref_scaff_dict = ref_scaff_count.to_dict()['Reference_Ligand']

In [298]:
results_df['N_Refs'] = results_df['Scaffold'].apply(lambda x: ref_scaff_dict[int(x)])

In [299]:
results_df

,Bootstraps,StructureChoice,StructureChoice_Choose_N,Score,Score_Choose_N,EvaluationMetric,EvaluationMetric_Cutoff,Split,N_Per_Split,Query_Scaffold_ID_Column,...,PoseSelection,PoseSelection_Choose_N,Min,Max,CI_Upper,CI_Lower,Total,Fraction,Scaffold,N_Refs
8,1000,Dock_to_All,All,POSIT,1,RMSD,2.0,ScaffoldSplit,5,cluster_id,...,Default,1,0.697368,0.697368,0.764773,0.620045,152,0.697368,0,14
1,1000,Dock_to_All,All,POSIT,1,RMSD,2.0,ScaffoldSplit,5,cluster_id,...,Default,1,0.717593,0.717593,0.773392,0.654054,216,0.717593,1,56
5,1000,Dock_to_All,All,POSIT,1,RMSD,2.0,ScaffoldSplit,5,cluster_id,...,Default,1,0.619266,0.619266,0.681149,0.553179,218,0.619266,10,7
4,1000,Dock_to_All,All,POSIT,1,RMSD,2.0,ScaffoldSplit,5,cluster_id,...,Default,1,0.000000,0.000000,0.016703,0.000116,218,0.000000,11,6
0,1000,Dock_to_All,All,POSIT,1,RMSD,2.0,ScaffoldSplit,5,cluster_id,...,Default,1,0.004587,0.004587,0.025177,0.001108,218,0.004587,12,6
2,1000,Dock_to_All,All,POSIT,1,RMSD,2.0,ScaffoldSplit,5,cluster_id,...,Default,1,0.706215,0.706215,0.768341,0.635165,177,0.706215,2,16
6,1000,Dock_to_All,All,POSIT,1,RMSD,2.0,ScaffoldSplit,5,cluster_id,...,Default,1,0.252294,0.252294,0.314005,0.199312,218,0.252294,3,17
9,1000,Dock_to_All,All,POSIT,1,RMSD,2.0,ScaffoldSplit,5,cluster_id,...,Default,1,0.739535,0.739535,0.793591,0.676921,215,0.739535,4,6
3,1000,Dock_to_All,All,POSIT,1,RMSD,2.0,ScaffoldSplit,5,cluster_id,...,Default,1,0.178899,0.178899,0.235303,0.133809,218,0.178899,6,8
10,1000,Dock_to_All,All,POSIT,1,RMSD,2.0,ScaffoldSplit,5,cluster_id,...,Default,1,0.000000,0.000000,0.016703,0.000116,218,0.000000,7,8


In [300]:
import plotly.express as px

In [301]:
results_df.sort_values(['Scaffold'], inplace=True)

In [303]:
plot_df = results_df.copy()
plot_df['Scaffold'] = results_df['Scaffold'].apply(lambda x: str(x))
px.scatter(plot_df, 
           x='N_Refs', 
           y='Fraction', 
           color='Scaffold', 
           size='Total', 
           template='simple_white', 
           width=800, 
           height=400)

/Users/alexpayne/miniforge-pypy3/envs/harbor/lib/python3.11/site-packages/plotly/express/_core.py:2065: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.



In [223]:
df.sort_values('cluster_id', inplace=True)

for cluster_id in df.cluster_id.unique():
    print(cluster_id, len(df[df.cluster_id == cluster_id].groupby('Query_Ligand').count()), len(df[df.cluster_id_Reference == cluster_id].groupby('Reference_Ligand').count()))

0 66 14
1 2 56
2 41 16
4 3 6
5 9 0
8 1 7
9 8 0
13 6 1
18 3 2
19 2 3
24 1 3
25 4 0
26 4 0
27 4 0
28 4 0
35 3 0
36 3 0
39 1 1
44 1 1
48 1 1
49 1 1
50 1 1
53 2 0
54 2 0
55 2 0
56 2 0
57 2 0
58 2 0
59 2 0
60 2 0
61 2 0
137 1 0
138 1 0
139 1 0
140 1 0
141 1 0
142 1 0
143 1 0
144 1 0
145 1 0
146 1 0
147 1 0
148 1 0
149 1 0
150 1 0
151 1 0
152 1 0
153 1 0
154 1 0
155 1 0
156 1 0
157 1 0
158 1 0
159 1 0
160 1 0
161 1 0
162 1 0
163 1 0
164 1 0
165 1 0
166 1 0
167 1 0


In [197]:
get_scaffold_counts(df, 'Query_Ligand', 'cluster_id')

,Query_Ligand
cluster_id,
0,66
1,2
2,41
4,3
5,9
...,...
163,1
164,1
165,1


# Plot just the query scaffolds with increasing number of reference scaffolds

In [72]:
results

,Unnamed: 0,Query_Ligand,Reference_Structure,Reference_Ligand_SMILES,SMILES,docking-confidence-POSIT,RMSD,Pose_ID,POSIT_Method,Reference_Ligand,...,bitsize,fingerprint,compound_name,cluster_id,scaffold_smarts,cluster_type,compound_name_Reference,cluster_id_Reference,scaffold_smarts_Reference,cluster_type_Reference
743687,743687,ALP-POS-6f6ae286-5,Mpro-x2563_0A,c1ccc(cc1)NC(=O)Cc2cncc3c2cccc3,CNC(=O)C[N@H+]1Cc2ccc(cc2[C@@H](C1)C(=O)Nc3cnc...,0.33,3.557773,0,HYBRID,DAR-DIA-23aa0b97-20,...,2048.0,ECFP4_2048,ALP-POS-6f6ae286-5,0,CC(CC1CCCC2CCCCC21)C1CCCC2CCCCC21,generic_bemis_murko,DAR-DIA-23aa0b97-20,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko
743318,743318,ALP-POS-6f6ae286-5,Mpro-x12677_0A,c1ccc2c(c1)cncc2NC(=O)Cc3cccc(n3)Cl,CNC(=O)C[N@H+]1Cc2ccc(cc2[C@@H](C1)C(=O)Nc3cnc...,0.42,0.839988,0,HYBRID,MAT-POS-afd4d4fd-2,...,2048.0,ECFP4_2048,ALP-POS-6f6ae286-5,0,CC(CC1CCCC2CCCCC21)C1CCCC2CCCCC21,generic_bemis_murko,MAT-POS-afd4d4fd-2,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko
743203,743203,ALP-POS-6f6ae286-5,Mpro-x11508_0A,c1ccc2c(c1)cnnc2NC(=O)Cc3cccc(c3)Cl,CNC(=O)C[N@H+]1Cc2ccc(cc2[C@@H](C1)C(=O)Nc3cnc...,0.50,1.687505,0,HYBRID,EDJ-MED-50fe53e8-1,...,2048.0,ECFP4_2048,ALP-POS-6f6ae286-5,0,CC(CC1CCCC2CCCCC21)C1CCCC2CCCCC21,generic_bemis_murko,EDJ-MED-50fe53e8-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko
743148,743148,ALP-POS-6f6ae286-5,Mpro-x11810_0A,c1ccc2c(c1)cncc2NC(=O)Cc3cc(ccn3)Cl,CNC(=O)C[N@H+]1Cc2ccc(cc2[C@@H](C1)C(=O)Nc3cnc...,0.42,0.957008,0,HYBRID,PET-UNK-3c72d439-1,...,2048.0,ECFP4_2048,ALP-POS-6f6ae286-5,0,CC(CC1CCCC2CCCCC21)C1CCCC2CCCCC21,generic_bemis_murko,PET-UNK-3c72d439-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko
744611,744611,ALP-POS-6f6ae286-5,Mpro-x11530_0A,c1cc(cc(c1)Cl)CC(=O)Nc2cncc3c2nccc3,CNC(=O)C[N@H+]1Cc2ccc(cc2[C@@H](C1)C(=O)Nc3cnc...,0.50,0.770488,0,HYBRID,MAT-POS-bb423b95-2,...,2048.0,ECFP4_2048,ALP-POS-6f6ae286-5,0,CC(CC1CCCC2CCCCC21)C1CCCC2CCCCC21,generic_bemis_murko,MAT-POS-bb423b95-2,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4552285,4552285,EDJ-MED-611d11e7-4,Mpro-x12677_0A,c1ccc2c(c1)cncc2NC(=O)Cc3cccc(n3)Cl,CO[C@]1(CCOc2c1cc(cc2)Cl)C(=O)Nc3cncc4c3cc(cc4)F,0.72,0.904653,0,SHAPEFIT,MAT-POS-afd4d4fd-2,...,2048.0,ECFP4_2048,EDJ-MED-611d11e7-4,0,CC(CC1CCCC2CCCCC21)C1CCCC2CCCCC21,generic_bemis_murko,MAT-POS-afd4d4fd-2,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko
4986793,4986793,MAT-POS-5cd9ea36-17,Mpro-x11499_0A,c1cc(cc(c1)Cl)CC(=O)Nc2cncc3c2cncc3,CCNS(=O)(=O)N1Cc2ccc(cc2[C@@H](C1)C(=O)Nc3cncc...,0.42,2.178248,0,HYBRID,MAT-POS-f7918075-5,...,2048.0,ECFP4_2048,MAT-POS-5cd9ea36-17,0,CC(CC1CCCC2CCCCC21)C1CCCC2CCCCC21,generic_bemis_murko,MAT-POS-f7918075-5,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko
4550976,4550976,EDJ-MED-611d11e7-4,Mpro-x10959_0A,c1ccc2c(c1)cncc2NC(=O)Cc3cccc(c3)Cl,CO[C@]1(CCOc2c1cc(cc2)Cl)C(=O)Nc3cncc4c3cc(cc4)F,0.72,1.239841,0,SHAPEFIT,ADA-UCB-6c2cb422-1,...,2048.0,ECFP4_2048,EDJ-MED-611d11e7-4,0,CC(CC1CCCC2CCCCC21)C1CCCC2CCCCC21,generic_bemis_murko,ADA-UCB-6c2cb422-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko
4553632,4553632,EDJ-MED-611d11e7-4,Mpro-x11530_0A,c1cc(cc(c1)Cl)CC(=O)Nc2cncc3c2nccc3,CO[C@]1(CCOc2c1cc(cc2)Cl)C(=O)Nc3cncc4c3cc(cc4)F,0.72,0.980561,0,SHAPEFIT,MAT-POS-bb423b95-2,...,2048.0,ECFP4_2048,EDJ-MED-611d11e7-4,0,CC(CC1CCCC2CCCCC21)C1CCCC2CCCCC21,generic_bemis_murko,MAT-POS-bb423b95-2,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko


In [372]:
from enum import Flag, auto
from pydantic import BaseModel

In [368]:
class TestFlag(Flag):
    A = auto()
    B = auto()
    C = auto()


In [369]:
TestFlag.A | TestFlag.B

<TestFlag.A|B: 3>

In [373]:
class TestModel(BaseModel):
    flag: TestFlag

In [378]:
tm = TestModel(flag=TestFlag.A | TestFlag.B)

In [383]:
TestFlag.A in tm.flag

True

In [397]:
import yaml

In [398]:
my_dict = {'bad_floats': [0.0]}

In [399]:
with open('test.yml', 'w') as f:
    yaml.dump(my_dict, f)

In [413]:
reload(cd)
s = cd.Settings()
s.to_yml_file('test.yml')

'test.yml'

# Test Compound Split Code

In [420]:
reload(cd)

<module 'harbor.analysis.cross_docking' from '/Users/alexpayne/Scientific_Projects/harbor/harbor/analysis/cross_docking.py'>

In [421]:
data = {"Split": [5], "Split": [10]}

In [426]:
sim_split

SimilaritySplit(type_='SimilaritySplit', name='SimilaritySplit', n_splits=1, n_per_split=100, deterministic=False, similarity_column='Tanimoto', groupby={'Aligned': True, 'Type': 'TanimotoCombo'}, query_ligand_column='Query_Ligand', threshold=0.5, higher_is_more_similar=True, include_similar=False)

In [437]:
settings = cd.Settings()
date_dict_list = (df.groupby(settings.reference_structure_column)[
[
    settings.reference_structure_column,
    settings.reference_structure_date_column,
]
].head(1).to_dict(orient="records"))

simplified_date_dict = {date_dict[settings.reference_structure_column]: date_dict[settings.reference_structure_date_column]for date_dict in date_dict_list}

In [472]:
reload(cd)
datesplit = cd.DateSplit(date_dict=simplified_date_dict, n_per_split=20, reference_structure_column=settings.reference_structure_column)

In [473]:
simsplit = cd.SimilaritySplit(n_per_split=5, similarity_column='Tanimoto', query_ligand_column='Query_Ligand', groupby={'Aligned': True, 'Type': 'TanimotoCombo'}, include_similar=False)

In [474]:
df1 = datesplit.run(df)

In [475]:
df2 = simsplit.run(df1[0])

In [476]:
test_df = df2[0]

In [477]:
reload(cd)

<module 'harbor.analysis.cross_docking' from '/Users/alexpayne/Scientific_Projects/harbor/harbor/analysis/cross_docking.py'>

In [478]:
scorer = cd.POSITScorer()
scored = scorer.run(test_df, groupby=['Query_Ligand'])

In [479]:
evaluator = cd.BinaryEvaluation(variable="RMSD", cutoff=2)
evaluated = evaluator.run(scored, groupby=['Query_Ligand'])

In [482]:
evaluated

FractionGood(type_='FractionGood', name='FractionGood', total=218, fraction=0.01834862385321101, replicates=[])

In [511]:
scored

,Unnamed: 0,Query_Ligand,Reference_Structure,Reference_Ligand_SMILES,SMILES,docking-confidence-POSIT,RMSD,Pose_ID,POSIT_Method,Reference_Ligand,...,bitsize,fingerprint,compound_name,cluster_id,scaffold_smarts,cluster_type,compound_name_Reference,cluster_id_Reference,scaffold_smarts_Reference,cluster_type_Reference
1325108,1325108,RAL-THA-2d450e86-7,Mpro-x0540_0A,c1cnccc1CCNC(=O)NC2CCCCC2,c1ccc2c(c1)cncc2NC(=O)Cc3ccc(cc3)F,0.55,8.728385,0,HYBRID,AAR-POS-d2a4d1df-12,...,NaN,None,RAL-THA-2d450e86-7,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko,AAR-POS-d2a4d1df-12,72,CC(CCCC1CCCCC1)CC1CCCCC1,generic_bemis_murko
3714448,3714448,RAL-THA-2d450e86-6,Mpro-x0540_0A,c1cnccc1CCNC(=O)NC2CCCCC2,c1ccc2c(c1)cncc2NC(=O)Cc3ccc(c(c3)Cl)F,0.55,8.870263,0,HYBRID,AAR-POS-d2a4d1df-12,...,NaN,None,RAL-THA-2d450e86-6,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko,AAR-POS-d2a4d1df-12,72,CC(CCCC1CCCCC1)CC1CCCCC1,generic_bemis_murko
2160748,2160748,MAT-POS-90fd5f68-7,Mpro-x0540_0A,c1cnccc1CCNC(=O)NC2CCCCC2,CN(C)c1ccc2c(c1)cncc2NC(=O)Cc3cccc(c3)Cl,0.50,9.052553,0,HYBRID,AAR-POS-d2a4d1df-12,...,NaN,None,MAT-POS-90fd5f68-7,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko,AAR-POS-d2a4d1df-12,72,CC(CCCC1CCCCC1)CC1CCCCC1,generic_bemis_murko
4214330,4214330,MIC-UNK-91acba05-6,Mpro-x0426_0A,c1ccc(c(c1)C(=O)NCCc2ccncc2)F,CN1CC[C@H](c2c1ccc(c2)Cl)C(=O)Nc3cncc4c3cccc4,0.42,8.469162,1,SHAPEFIT,AAR-POS-d2a4d1df-10,...,NaN,None,MIC-UNK-91acba05-6,0,CC(CC1CCCC2CCCCC21)C1CCCC2CCCCC21,generic_bemis_murko,AAR-POS-d2a4d1df-10,134,CC(CCCC1CCCCC1)C1CCCCC1,generic_bemis_murko
1778362,1778362,EDG-MED-5d232de5-6,Mpro-x0107_0A,Cc1ccncc1NC(=O)C,CN1CC[C@H](c2c1ccc(c2)Cl)C(=O)Nc3cncc4c3cccc4,0.42,3.822589,0,SHAPEFIT,MAK-UNK-6435e6c2-8,...,NaN,None,EDG-MED-5d232de5-6,0,CC(CC1CCCC2CCCCC21)C1CCCC2CCCCC21,generic_bemis_murko,MAK-UNK-6435e6c2-8,3,C1CCCCC1,generic_bemis_murko
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3675355,3675355,EDJ-MED-76744c27-4,Mpro-x0305_0A,CCNc1ccc(cn1)C#N,COC1(CC1)CS(=O)(=O)N2Cc3ccc(cc3[C@@H](C2)C(=O)...,0.02,6.299657,13,FRED,AAR-POS-d2a4d1df-5,...,NaN,None,EDJ-MED-76744c27-4,27,CC(CC1CCCC2CCCCC21)C1CC(C(C)(C)CC2CC2)CC2CCCCC21,generic_bemis_murko,AAR-POS-d2a4d1df-5,3,C1CCCCC1,generic_bemis_murko
716508,716508,BRU-THA-92256091-17,Mpro-x0354_0A,Cc1ccc(cc1)OCC(=O)N2CC[NH+](CC2)C,[H:25][n+]1cc([nH]c1)C(=O)N(c2ccc(cc2)OC(C)C)[...,0.02,7.703777,3,FRED,AAR-POS-d2a4d1df-6,...,NaN,None,BRU-THA-92256091-17,57,CC(CCCC1CCCCC1)C(C1CCCCC1)C(C1CCCCC1)C(C)C1CCCC1,generic_bemis_murko,AAR-POS-d2a4d1df-6,45,CC(CCC1CCCCC1)C1CCCCC1,generic_bemis_murko
519784,519784,MAT-POS-4223bc15-28,Mpro-x0689_0A,CC(=O)N1CCN(CC1)S(=O)(=O)c2cccs2,COCC[N@H+]1Cc2ccc(cc2[C@@H](C1)C(=O)Nc3cncc4c3...,0.02,7.179341,1,FRED,LON-WEI-8f408cad-3,...,NaN,None,MAT-POS-4223bc15-28,0,CC(CC1CCCC2CCCCC21)C1CCCC2CCCCC21,generic_bemis_murko,LON-WEI-8f408cad-3,37,CC(C)(C1CCCCC1)C1CCCC1,generic_bemis_murko
4117116,4117116,EDG-MED-ba1ac7b9-19,Mpro-x0705_0A,Cc1ccc(cc1)C(=O)N2CCN(CC2)C(=O)C,C[C@H]1c2nncn2CCN1C(=O)C[C@]3(CCOc4c3cc(cc4)Cl...,0.02,5.477716,0,FRED,AAR-POS-d2a4d1df-25,...,NaN,None,EDG-MED-ba1ac7b9-19,139,CC(CC1(C(C)CC2CCCC3CCCCC32)CCCC2CCCCC21)C1CCC2...,generic_bemis_murko,AAR-POS-d2a4d1df-25,33,CC(C1CCCCC1)C1CCCCC1,generic_bemis_murko


In [512]:
datesplit = cd.DateSplit(date_dict=simplified_date_dict,
                                             n_per_split=20, reference_structure_column=settings.reference_structure_column)
simsplit = cd.SimilaritySplit(n_per_split=5, similarity_column='Tanimoto', query_ligand_column='Query_Ligand', groupby={'Aligned': True, 'Type': 'TanimotoCombo'}, include_similar=False)

In [513]:
df

,Unnamed: 0,Query_Ligand,Reference_Structure,Reference_Ligand_SMILES,SMILES,docking-confidence-POSIT,RMSD,Pose_ID,POSIT_Method,Reference_Ligand,...,bitsize,fingerprint,compound_name,cluster_id,scaffold_smarts,cluster_type,compound_name_Reference,cluster_id_Reference,scaffold_smarts_Reference,cluster_type_Reference
0,0,ERI-UCB-d6de1f3c-2,Mpro-x11810_0A,c1ccc2c(c1)cncc2NC(=O)Cc3cc(ccn3)Cl,c1ccc2c(c1)cncc2C(=O)N3CCN(C(=O)C3)c4cccc(c4)Cl,0.72,1.427564,0,SHAPEFIT,PET-UNK-3c72d439-1,...,NaN,None,ERI-UCB-d6de1f3c-2,48,CC1CC(C(C)C2CCCC3CCCCC32)CCC1C1CCCCC1,generic_bemis_murko,PET-UNK-3c72d439-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko
1,1,ERI-UCB-d6de1f3c-2,Mpro-x11810_0A,c1ccc2c(c1)cncc2NC(=O)Cc3cc(ccn3)Cl,c1ccc2c(c1)cncc2C(=O)N3CCN(C(=O)C3)c4cccc(c4)Cl,0.72,1.427564,0,SHAPEFIT,PET-UNK-3c72d439-1,...,NaN,None,ERI-UCB-d6de1f3c-2,48,CC1CC(C(C)C2CCCC3CCCCC32)CCC1C1CCCCC1,generic_bemis_murko,PET-UNK-3c72d439-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko
2,2,ERI-UCB-d6de1f3c-2,Mpro-x11810_0A,c1ccc2c(c1)cncc2NC(=O)Cc3cc(ccn3)Cl,c1ccc2c(c1)cncc2C(=O)N3CCN(C(=O)C3)c4cccc(c4)Cl,0.72,1.427564,0,SHAPEFIT,PET-UNK-3c72d439-1,...,NaN,None,ERI-UCB-d6de1f3c-2,48,CC1CC(C(C)C2CCCC3CCCCC32)CCC1C1CCCCC1,generic_bemis_murko,PET-UNK-3c72d439-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko
3,3,ERI-UCB-d6de1f3c-2,Mpro-x11810_0A,c1ccc2c(c1)cncc2NC(=O)Cc3cc(ccn3)Cl,c1ccc2c(c1)cncc2C(=O)N3CCN(C(=O)C3)c4cccc(c4)Cl,0.72,1.427564,0,SHAPEFIT,PET-UNK-3c72d439-1,...,NaN,None,ERI-UCB-d6de1f3c-2,48,CC1CC(C(C)C2CCCC3CCCCC32)CCC1C1CCCCC1,generic_bemis_murko,PET-UNK-3c72d439-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko
4,4,ERI-UCB-d6de1f3c-2,Mpro-x11810_0A,c1ccc2c(c1)cncc2NC(=O)Cc3cc(ccn3)Cl,c1ccc2c(c1)cncc2C(=O)N3CCN(C(=O)C3)c4cccc(c4)Cl,0.72,1.427564,0,SHAPEFIT,PET-UNK-3c72d439-1,...,NaN,None,ERI-UCB-d6de1f3c-2,48,CC1CC(C(C)C2CCCC3CCCCC32)CCC1C1CCCCC1,generic_bemis_murko,PET-UNK-3c72d439-1,2,CC(CC1CCCCC1)CC1CCCC2CCCCC21,generic_bemis_murko
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5599424,5599424,MAT-POS-e6dd326d-8,Mpro-x12300_0A,Cc1ccncc1NC(=O)Cc2cc(cc(c2)Cl)NCC(C)(C)C#N,C=CC(=O)NC[C@]1(CCOc2c1cc(cc2)Cl)C(=O)Nc3cncc4...,0.05,6.445683,48,FRED,MAT-POS-044491d2-7,...,2048.0,ECFP6_2048,MAT-POS-e6dd326d-8,0,CC(CC1CCCC2CCCCC21)C1CCCC2CCCCC21,generic_bemis_murko,MAT-POS-044491d2-7,1,CC(CC1CCCCC1)CC1CCCCC1,generic_bemis_murko
5599425,5599425,MAT-POS-e6dd326d-8,Mpro-x12300_0A,Cc1ccncc1NC(=O)Cc2cc(cc(c2)Cl)NCC(C)(C)C#N,C=CC(=O)NC[C@]1(CCOc2c1cc(cc2)Cl)C(=O)Nc3cncc4...,0.05,6.445683,48,FRED,MAT-POS-044491d2-7,...,1024.0,ECFP8_1024,MAT-POS-e6dd326d-8,0,CC(CC1CCCC2CCCCC21)C1CCCC2CCCCC21,generic_bemis_murko,MAT-POS-044491d2-7,1,CC(CC1CCCCC1)CC1CCCCC1,generic_bemis_murko
5599426,5599426,MAT-POS-e6dd326d-8,Mpro-x12300_0A,Cc1ccncc1NC(=O)Cc2cc(cc(c2)Cl)NCC(C)(C)C#N,C=CC(=O)NC[C@]1(CCOc2c1cc(cc2)Cl)C(=O)Nc3cncc4...,0.05,6.445683,48,FRED,MAT-POS-044491d2-7,...,2048.0,ECFP8_2048,MAT-POS-e6dd326d-8,0,CC(CC1CCCC2CCCCC21)C1CCCC2CCCCC21,generic_bemis_murko,MAT-POS-044491d2-7,1,CC(CC1CCCCC1)CC1CCCCC1,generic_bemis_murko
5599427,5599427,MAT-POS-e6dd326d-8,Mpro-x12300_0A,Cc1ccncc1NC(=O)Cc2cc(cc(c2)Cl)NCC(C)(C)C#N,C=CC(=O)NC[C@]1(CCOc2c1cc(cc2)Cl)C(=O)Nc3cncc4...,0.05,6.445683,48,FRED,MAT-POS-044491d2-7,...,1024.0,ECFP10_1024,MAT-POS-e6dd326d-8,0,CC(CC1CCCC2CCCCC21)C1CCCC2CCCCC21,generic_bemis_murko,MAT-POS-044491d2-7,1,CC(CC1CCCCC1)CC1CCCCC1,generic_bemis_murko


In [480]:
scored[scored.RMSD < 2]

,Unnamed: 0,Query_Ligand,Reference_Structure,Reference_Ligand_SMILES,SMILES,docking-confidence-POSIT,RMSD,Pose_ID,POSIT_Method,Reference_Ligand,...,bitsize,fingerprint,compound_name,cluster_id,scaffold_smarts,cluster_type,compound_name_Reference,cluster_id_Reference,scaffold_smarts_Reference,cluster_type_Reference
2143648,2143648,EDJ-MED-43f8f7d6-4,Mpro-x0678_0A,c1cc(cnc1)NC(=O)CC2CCCCC2,c1ccc2c(c1)cncc2NC(=O)[C@@H]3CN(C(=O)c4c3cc(cc...,0.42,1.062151,0,HYBRID,ALE-HEI-f28a35b5-9,...,NaN,None,EDJ-MED-43f8f7d6-4,143,CC(CC1CC1)CC1CC(C(C)CC2CCCC3CCCCC32)C2CCCCC2C1C,generic_bemis_murko,ALE-HEI-f28a35b5-9,1,CC(CC1CCCCC1)CC1CCCCC1,generic_bemis_murko
774981,774981,EDJ-MED-8bb691af-8,Mpro-x0678_0A,c1cc(cnc1)NC(=O)CC2CCCCC2,CNC(=O)CN1C[C@]2(CCN(C2=O)c3cncc4c3cccc4)c5cc(...,0.33,0.771036,0,HYBRID,ALE-HEI-f28a35b5-9,...,NaN,None,EDJ-MED-8bb691af-8,54,CC1CCC2(CCC(C3CCCC4CCCCC43)C2C)C2CCCCC12,generic_bemis_murko,ALE-HEI-f28a35b5-9,1,CC(CC1CCCCC1)CC1CCCCC1,generic_bemis_murko
3876992,3876992,MAT-POS-e69ad64a-2,Mpro-x0434_0A,c1ccc(cc1)NC(=O)Nc2cccnc2,CCC(=O)N(c1cncc2c1cccc2)C(=O)[C@@H]3COc4c3cc(c...,0.33,1.259592,0,HYBRID,AAR-POS-d2a4d1df-11,...,NaN,None,MAT-POS-e69ad64a-2,59,CC(CC1CCCC2CCCCC21)C1CCC2CCCCC21,generic_bemis_murko,AAR-POS-d2a4d1df-11,1,CC(CC1CCCCC1)CC1CCCCC1,generic_bemis_murko
4009122,4009122,EDJ-MED-2f867453-1,Mpro-x0708_0A,CC(=O)NNC(=O)c1cc2c(s1)CCCC2,C[C@]1(CNc2c1cc(cc2)Cl)C(=O)Nc3cncc4c3cccc4,0.24,1.534175,0,FRED,AAR-POS-d2a4d1df-26,...,NaN,None,EDJ-MED-2f867453-1,59,CC(CC1CCCC2CCCCC21)C1CCC2CCCCC21,generic_bemis_murko,AAR-POS-d2a4d1df-26,15,C1CCC2CCCC2C1,generic_bemis_murko


In [551]:
reload(cd)
ev = cd.Evaluator(dataset_split=cd.DateSplit(date_dict=simplified_date_dict,
                                             n_per_split=20, reference_structure_column=settings.reference_structure_column),
                  extra_splits=[cd.ScaffoldSplit(n_per_split=5, query_scaffold_id_column='cluster_id',
                                             reference_scaffold_id_column='cluster_id_Reference',      
                                             split_option=cd.ScaffoldSplitOptions.NOT_X_TO_X,
                                                 reference_scaffold_id_subset=[0])],
                  scorer=cd.POSITScorer(),
                  evaluator=cd.BinaryEvaluation(variable="RMSD", cutoff=2),n_bootstraps=1,
                  groupby=['Query_Ligand'])

In [552]:
ev.to_json_file('test.json')

'test.json'

In [524]:
results = cd.Results(evaluator=ev, fraction_good=ev.run(df))

In [525]:
test_out_df = cd.Results.df_from_results([results])

In [526]:
test_out_df

,Bootstraps,StructureChoice,StructureChoice_Choose_N,Score,Score_Choose_N,EvaluationMetric,EvaluationMetric_Cutoff,Split,N_Per_Split,Reference_Structure_Column,...,Include_Similar_1,Higher_Is_More_Similar_1,Aligned_1,Type_1,Min,Max,CI_Upper,CI_Lower,Total,Fraction
0,1,Dock_to_All,All,POSIT,1,RMSD,2.0,DateSplit,20,Reference_Structure,...,False,True,True,TanimotoCombo,0.0625,0.0625,0.104017,0.037103,208,0.0625


In [543]:
reload(cd)
settings = cd.Settings()

In [544]:
settings.to_yml_file('test.yml')

RepresenterError: ('cannot represent an object', <ScaffoldSplitOptions.X_TO_X: 'x_to_x'>)